# Проект для "Викишоп"

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Подготовка

In [39]:
import pandas as pd
import numpy as np
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Egor_Aristov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
RANDOM_STATE = 42

### Загрузка данных

In [3]:
data = pd.read_csv('toxic_comments.csv', index_col=0)
data = data.reset_index(drop=True)

In [4]:
data.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.4+ MB


### Очистка данных

Оставим в текстах только латинские буквы и пробелы с помощью регулярных выражений

In [6]:
# функция для очистки текста
def clear_text(text):
    return ' '.join(re.sub(r'[^a-zA-Z]', ' ', text).split())

In [7]:
data['text'] = data['text'].apply(clear_text)

In [9]:
data.head()

,text,toxic
0,Explanation Why the edits made under my userna...,0
1,D aww He matches this background colour I m se...,0
2,Hey man I m really not trying to edit war It s...,0
3,More I can t make any real suggestions on impr...,0
4,You sir are my hero Any chance you remember wh...,0


### Лемматизация

In [18]:
nlp = spacy.load("en_core_web_sm")

In [19]:
# функция для лемматизации
def lemmatize_text(text):
    return ' '.join([i.lemma_ for i in nlp(text)])

In [16]:
%%time
data['text'] = data['text'].apply(lemmatize_text)

CPU times: total: 38min 56s
Wall time: 38min 57s


### TF-IDF

Разделим выборку на тренировочную и тестовую

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    data['text'],
    data['toxic'],
    random_state = RANDOM_STATE
)

In [28]:
tfidf_vec = TfidfVectorizer(stop_words = list(set(stopwords.words('english'))))

X_train = tfidf_vec.fit_transform(X_train)

X_test = tfidf_vec.transform(X_test)

## Обучение

In [29]:
main_pipe = Pipeline([
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

In [30]:
param_grid = [
    # словарь для LogisticRegression()
    {
        'model' : [LogisticRegression()],
        'model__max_iter' : [80, 100, 150],
        'model__C' : [.5, 1., 2.]
    },

    # словарь для KNeighborsClassifier()
    {
        'model' : [KNeighborsClassifier()],
        'model__n_neighbors' : range(2, 25)
    },

    # словарь для SVC()
    {
        'model' : [SVC()],
        'model__C' : [0.1, 1, 10, 100]
    }
]

In [32]:
%%time
randomized_search = RandomizedSearchCV(
    main_pipe,
    param_grid,
    n_jobs=-1,
    scoring='f1',
    random_state=RANDOM_STATE
)
randomized_search.fit(X_train, y_train)

CPU times: total: 51min 30s
Wall time: 1h 55min 18s


RandomizedSearchCV(estimator=Pipeline(steps=[('model',
                                              LogisticRegression(random_state=42))]),
                   n_jobs=-1,
                   param_distributions=[{'model': [LogisticRegression()],
                                         'model__C': [0.5, 1.0, 2.0],
                                         'model__max_iter': [80, 100, 150]},
                                        {'model': [KNeighborsClassifier()],
                                         'model__n_neighbors': range(2, 25)},
                                        {'model': [SVC()],
                                         'model__C': [0.1, 1, 10, 100]}],
                   random_state=42, scoring='f1')

In [33]:
randomized_search.best_score_

np.float64(0.7583041668911579)

Проверим модель на тестовой выборке

In [37]:
f1_score(y_test, randomized_search.best_estimator_.predict(X_test))

0.7837990403612758

## Вывод

По полученным данным была проведена следующая работа

**Подготовка**

- Данные загружены
- В текстах оставлены только латинские символы и пробелы
- Текст лемматизирован
- Составлены матрицы TF-IDF

**Обучение**

С использованием пайплайнов обучена модель SVC с метрико F1 на тестовой выборке равной 0.78